# Week 5: Ensemble Learning & Decision Trees
## Notebook 07: Stacking & Blending Ensembles from Scratch

### 1. Conceptual Intuition & Real-World Use Case
In machine learning competitions (Kaggle) and enterprise forecasting systems:
- Individual models have distinct inductive biases: Linear Models excel at linear trends, Random Forests capture robust feature thresholds, and Gradient Boosted Trees capture complex non-linear interactions.
- Simple averaging treats all models equally, even when one model is vastly superior or makes redundant errors.

**Stacking (Stacked Generalization)** trains a **Meta-Learner** (e.g. Ridge Regression or Logistic Regression) to dynamically weight and combine the predictions of multiple base models!

---

### 2. Mathematical Foundations: Out-of-Fold (OOF) Stacking
If you train base models on the full dataset and feed their predictions to the meta-learner, **catastrophic overfitting occurs** because the meta-learner learns to trust whichever base model overfit the training data most.

#### Out-of-Fold (OOF) Protocol:
1. Split training set into $K$ folds.
2. For fold $k$:
   - Train Base Models on the remaining $K-1$ folds.
   - Predict on the holdout fold $k$.
3. Concatenate the $K$ holdout predictions to form the **Meta-Feature Matrix** $S \in \mathbb{R}^{N \times M}$ (where $M$ is the number of base models).
4. Train the Meta-Learner on $S$ against the true target $y$.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

# Step 1: Create Complex Synthetic Dataset
X, y = make_classification(n_samples=600, n_features=12, n_informative=8, random_state=42)

# Step 2: Define Diverse Base Models
base_models = {
    "LogisticRegression": LogisticRegression(),
    "RandomForest": RandomForestClassifier(n_estimators=30, max_depth=4, random_state=42),
    "Ridge": RidgeClassifier()
}

# Step 3: Implement Out-of-Fold Stacking Engine
def generate_oof_features(base_models, X, y, n_splits=5):
    N = len(X)
    num_models = len(base_models)
    oof_predictions = np.zeros((N, num_models))
    
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        X_train, y_train = X[train_idx], y[train_idx]
        X_val = X[val_idx]
        
        for m_idx, (name, model) in enumerate(base_models.items()):
            # Clone model logic by fitting a new instance
            model_clone = model.__class__(**model.get_params())
            model_clone.fit(X_train, y_train)
            
            # Predict probabilities or continuous decision scores
            if hasattr(model_clone, "predict_proba"):
                preds = model_clone.predict_proba(X_val)[:, 1]
            else:
                preds = model_clone.decision_function(X_val)
                # Map to [0, 1] using sigmoid
                preds = 1.0 / (1.0 + np.exp(-preds))
                
            oof_predictions[val_idx, m_idx] = preds
            
    return oof_predictions

print("Generating Out-of-Fold Meta-Features across 5 Folds...")
oof_meta_features = generate_oof_features(base_models, X, y)
print("Meta-Feature Matrix Shape:", oof_meta_features.shape)

# Step 4: Train Meta-Learner (Logistic Regression) on OOF Predictions
meta_learner = LogisticRegression()
meta_learner.fit(oof_meta_features, y)

print("\n--- Meta-Learner Learned Model Weights (Coefficients) ---")
for name, weight in zip(base_models.keys(), meta_learner.coef_[0]):
    print(f"  {name:<20}: Weight = {weight:.3f}")

# Evaluate Individual Base Models vs Stacking Meta-Ensemble
for idx, name in enumerate(base_models.keys()):
    base_acc = accuracy_score(y, (oof_meta_features[:, idx] >= 0.5).astype(int))
    print(f"Base Model [{name}] CV Accuracy: {base_acc * 100:.2f}%")

ensemble_preds = meta_learner.predict(oof_meta_features)
ensemble_acc = accuracy_score(y, ensemble_preds)
print(f"\nStacking Meta-Ensemble CV Accuracy: {ensemble_acc * 100:.2f}%")

### 3. Step-by-Step Code Walkthrough
1. **Model Diversity:** We combined linear models and decision trees with different inductive biases.
2. **K-Fold OOF Protection:** Predictions in `oof_meta_features` were generated strictly by models that never saw those validation samples.
3. **Meta-Learner Synergy:** The meta-learner assigned higher weights to models that performed best, achieving higher overall accuracy than any individual base model.

---

### 4. Industry Applications & Trade-offs
- **Kaggle Winning Pipelines:** Stacking 20+ diverse neural networks and GBDTs.
- **Trade-off:** High operational latency and complexity: inference requires running all base models in parallel before invoking the meta-learner.

---

### 5. Top Interview Questions & Answers
**Q1: Why is a simple linear model usually preferred as a meta-learner over a complex tree model?**
> **Answer:** The meta-feature matrix contains only a few columns ($M \approx 3$ to $10$) representing model probabilities. Training a deep tree on these inputs quickly leads to severe overfitting. A simple regularized linear model (Ridge or Logistic Regression) acts as a smooth, robust combiner.

**Q2: What is the difference between Stacking and Blending?**
> **Answer:** Blending uses a single holdout validation split (e.g. 80% train, 20% validation) to generate meta-features, which is faster but discards 20% of data for base training. Stacking uses $K$-Fold cross-validation, ensuring 100% of the training dataset receives out-of-fold predictions.